# NLP Assignment 5: Fine-Tuning BERT for POS Tagging & Chunking

**Objective:** Build and fine-tune a transformer model (BERT/DistilBERT) to perform Part-of-Speech (POS) Tagging and Chunking (Phrase Detection) using token classification techniques.

**Pipeline:** Raw Data → Tokenization → Label Alignment → Model Training → Evaluation → Inference → Comparison

---

## Step 0: Install Dependencies

In [13]:
# Install required libraries
!pip install transformers datasets seqeval evaluate accelerate -q

In [14]:
!pip install -U transformers

In [27]:
!pip install transformers==4.38.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.7/130.7 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 71.9 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.8.0
    Uninstalling huggingface_hub-1.8.0:
      Successfully uninstalled huggingface_hub-1.8.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.5.2
    Uninstalling transformers-5.5.2:
      Successfully uninstalled transformers-5.5.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the follow

In [15]:
!pip install datasets==2.16.1

In [16]:
!pip uninstall -y transformers datasets evaluate peft accelerate
!pip install transformers==4.38.2 datasets==2.16.1 evaluate==0.4.1 peft==0.5.0 accelerate==0.27.2

Found existing installation: transformers 4.38.2
Uninstalling transformers-4.38.2:
  Successfully uninstalled transformers-4.38.2
Found existing installation: datasets 2.16.1
Uninstalling datasets-2.16.1:
  Successfully uninstalled datasets-2.16.1
Found existing installation: evaluate 0.4.1
Uninstalling evaluate-0.4.1:
  Successfully uninstalled evaluate-0.4.1
Found existing installation: peft 0.7.1
Uninstalling peft-0.7.1:
  Successfully uninstalled peft-0.7.1
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
  Using cached transformers-4.38.2-py3-none-any.whl.metadata (130 kB)
  Using cached datasets-2.16.1-py3-none-any.whl.metadata (20 kB)
  Using cached evaluate-0.4.1-py3-none-any.whl.metadata (9.4 kB)
Using cached transformers-4.38.2-py3-none-any.whl (8.5 MB)
Using cached datasets-2.16.1-py3-none-any.whl (507 kB)
Using cached evaluate-0.4.1-py3-none-any.whl (84 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## Task 1: Dataset Selection (10%)

**Dataset Chosen:** CoNLL-2003

The CoNLL-2003 dataset is a standard benchmark for Named Entity Recognition (NER) and Chunking tasks. It contains:
- **POS Tags:** Part-of-speech labels (NN, VBZ, IN, NNP, etc.)
- **Chunk Tags:** Phrase-level groupings using BIO scheme (B-NP, I-NP, B-VP, etc.)
- **NER Tags:** Entity labels (used only for context; we focus on POS + Chunk)

### Label Categories
- **POS Tags:** NN, NNS, NNP, NNPS, VB, VBD, VBG, VBN, VBP, VBZ, JJ, JJR, JJS, RB, RBR, RBS, IN, DT, CC, CD, PRP, PRP$, WP, WDT, MD, TO, EX, FW, LS, PDT, POS, RP, SYM, UH, VBP, WP$, WRB, etc.
- **Chunk Tags (BIO scheme):** B-NP, I-NP, B-VP, I-VP, B-PP, I-PP, B-ADJP, I-ADJP, B-ADVP, I-ADVP, B-SBAR, I-SBAR, B-PRT, I-PRT, B-CONJP, I-CONJP, B-INTJ, I-INTJ, B-LST, I-LST, B-UCP, I-UCP, O

In [4]:
# ─────────────────────────────────────────────
# TASK 1: Import Libraries & Load Dataset
# ─────────────────────────────────────────────

# Compatibility fix for transformers/peft is now handled in an earlier cell.

import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)
import evaluate
import warnings
warnings.filterwarnings('ignore')

# Load CoNLL-2003 dataset (contains POS tags, Chunk tags, and NER tags)


raw_dataset = load_dataset("conll2003")
print("Dataset loaded successfully!")
print(raw_dataset)

Dataset loaded successfully!
DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})


In [5]:
# Inspect the dataset structure
print("Features:", raw_dataset["train"].features)
print("\nSample record:")
sample = raw_dataset["train"][0]
print("Tokens:    ", sample["tokens"])
print("POS Tags:  ", sample["pos_tags"])
print("Chunk Tags:", sample["chunk_tags"])

Features: {'id': Value(dtype='string', id=None), 'tokens': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None), 'pos_tags': Sequence(feature=ClassLabel(names=['"', "''", '#', '$', '(', ')', ',', '.', ':', '``', 'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS', 'LS', 'MD', 'NN', 'NNP', 'NNPS', 'NNS', 'NN|SYM', 'PDT', 'POS', 'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'SYM', 'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'WDT', 'WP', 'WP$', 'WRB'], id=None), length=-1, id=None), 'chunk_tags': Sequence(feature=ClassLabel(names=['O', 'B-ADJP', 'I-ADJP', 'B-ADVP', 'I-ADVP', 'B-CONJP', 'I-CONJP', 'B-INTJ', 'I-INTJ', 'B-LST', 'I-LST', 'B-NP', 'I-NP', 'B-PP', 'I-PP', 'B-PRT', 'I-PRT', 'B-SBAR', 'I-SBAR', 'B-UCP', 'I-UCP', 'B-VP', 'I-VP'], id=None), length=-1, id=None), 'ner_tags': Sequence(feature=ClassLabel(names=['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC'], id=None), length=-1, id=None)}

Sample record:
Tokens:     ['EU', 'rejects'

In [6]:
# Extract POS and Chunk label lists from the dataset features
pos_label_list   = raw_dataset["train"].features["pos_tags"].feature.names
chunk_label_list = raw_dataset["train"].features["chunk_tags"].feature.names

print(f"POS label count:   {len(pos_label_list)}")
print("POS labels:  ", pos_label_list)

print(f"\nChunk label count: {len(chunk_label_list)}")
print("Chunk labels:", chunk_label_list)

POS label count:   47
POS labels:   ['"', "''", '#', '$', '(', ')', ',', '.', ':', '``', 'CC', 'CD', 'DT', 'EX', 'FW', 'IN', 'JJ', 'JJR', 'JJS', 'LS', 'MD', 'NN', 'NNP', 'NNPS', 'NNS', 'NN|SYM', 'PDT', 'POS', 'PRP', 'PRP$', 'RB', 'RBR', 'RBS', 'RP', 'SYM', 'TO', 'UH', 'VB', 'VBD', 'VBG', 'VBN', 'VBP', 'VBZ', 'WDT', 'WP', 'WP$', 'WRB']

Chunk label count: 23
Chunk labels: ['O', 'B-ADJP', 'I-ADJP', 'B-ADVP', 'I-ADVP', 'B-CONJP', 'I-CONJP', 'B-INTJ', 'I-INTJ', 'B-LST', 'I-LST', 'B-NP', 'I-NP', 'B-PP', 'I-PP', 'B-PRT', 'I-PRT', 'B-SBAR', 'I-SBAR', 'B-UCP', 'I-UCP', 'B-VP', 'I-VP']


---
## Task 2: Data Preprocessing (15%)

### Key Challenges:
1. **Subword tokenization:** BERT uses WordPiece tokenizer which can split one word into multiple subword tokens.
2. **Label alignment:** We must align the original word-level labels with the new subword-level tokens.
3. **Special tokens:** `[CLS]` and `[SEP]` tokens are added by BERT. We assign `-100` to them so they are ignored during loss computation.

In [7]:
# ─────────────────────────────────────────────
# TASK 2: Data Preprocessing
# ─────────────────────────────────────────────

# We will perform TWO separate fine-tuning runs:
#   (A) POS Tagging  – using pos_tags
#   (B) Chunking     – using chunk_tags

MODEL_CHECKPOINT = "distilbert-base-uncased"  # Lightweight and fast

# Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
print(f"Tokenizer loaded: {MODEL_CHECKPOINT}")

Tokenizer loaded: distilbert-base-uncased


In [8]:
def tokenize_and_align_labels(examples, label_column):
    """
    Tokenizes input sentences using BERT tokenizer and aligns labels
    to handle subword tokens and special tokens.

    Args:
        examples     : Batch of examples from HuggingFace dataset.
        label_column : Either 'pos_tags' or 'chunk_tags'.

    Returns:
        Tokenized inputs with aligned labels.
    """
    # Tokenize the words; is_split_into_words=True because input is already word-tokenized
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        is_split_into_words=True  # Important: tells tokenizer input is pre-tokenized
    )

    all_labels = []
    for i, labels in enumerate(examples[label_column]):
        # word_ids() maps each subword token back to the original word index
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            if word_idx is None:
                # Special tokens ([CLS], [SEP]) → assign -100 (ignored in loss)
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                # First subword of a word → assign the actual label
                label_ids.append(labels[word_idx])
            else:
                # Subsequent subwords of the same word → assign -100 (ignored)
                label_ids.append(-100)
            previous_word_idx = word_idx

        all_labels.append(label_ids)

    tokenized_inputs["labels"] = all_labels
    return tokenized_inputs

print("Label alignment function defined.")

Label alignment function defined.


In [9]:
# ── Preprocess for POS Tagging ──
pos_tokenized_dataset = raw_dataset.map(
    lambda examples: tokenize_and_align_labels(examples, label_column="pos_tags"),
    batched=True
)
print("POS Tagging dataset preprocessed!")

# ── Preprocess for Chunking ──
chunk_tokenized_dataset = raw_dataset.map(
    lambda examples: tokenize_and_align_labels(examples, label_column="chunk_tags"),
    batched=True
)
print("Chunking dataset preprocessed!")

Map:   0%|          | 0/3250 [00:00<?, ? examples/s]

POS Tagging dataset preprocessed!
Chunking dataset preprocessed!


In [10]:
# Verify output: check a sample tokenized example
sample_idx = 0
sample_pos = pos_tokenized_dataset["train"][sample_idx]

tokens    = tokenizer.convert_ids_to_tokens(sample_pos["input_ids"])
labels    = sample_pos["labels"]
attn_mask = sample_pos["attention_mask"]

print(f"{'Token':<20} {'Label ID':<12} {'Attn Mask'}")
print("-" * 45)
for tok, lbl, msk in zip(tokens, labels, attn_mask):
    lbl_str = pos_label_list[lbl] if lbl != -100 else "[IGNORED]"
    print(f"{tok:<20} {str(lbl_str):<12} {msk}")

Token                Label ID     Attn Mask
---------------------------------------------
[CLS]                [IGNORED]    1
eu                   NNP          1
rejects              VBZ          1
german               JJ           1
call                 NN           1
to                   TO           1
boycott              VB           1
british              JJ           1
lamb                 NN           1
.                    .            1
[SEP]                [IGNORED]    1


---
## Task 3: Model Setup (15%)

We use `AutoModelForTokenClassification` from Hugging Face:
- Adds a **token-level classification head** (linear layer) on top of DistilBERT.
- `num_labels` must match the number of unique labels in the chosen task.
- `id2label` and `label2id` mappings are set for interpretability.

In [11]:
# ─────────────────────────────────────────────
# TASK 3: Model Setup
# ─────────────────────────────────────────────

def build_model(label_list, model_checkpoint=MODEL_CHECKPOINT):
    """
    Builds a DistilBERT token classification model.

    Args:
        label_list        : List of label strings for the task.
        model_checkpoint  : Pre-trained model name from HuggingFace Hub.

    Returns:
        Initialized model with correct num_labels and id2label/label2id mappings.
    """
    # Create label ↔ id mappings
    id2label = {i: label for i, label in enumerate(label_list)}
    label2id = {label: i for i, label in enumerate(label_list)}

    model = AutoModelForTokenClassification.from_pretrained(
        model_checkpoint,
        num_labels=len(label_list),  # Number of unique output classes
        id2label=id2label,           # Maps prediction index → label string
        label2id=label2id            # Maps label string → index
    )
    return model, id2label, label2id


# Build models for both tasks
pos_model,   pos_id2label,   pos_label2id   = build_model(pos_label_list)
chunk_model, chunk_id2label, chunk_label2id = build_model(chunk_label_list)

print(f"POS Model   → num_labels: {pos_model.config.num_labels}")
print(f"Chunk Model → num_labels: {chunk_model.config.num_labels}")
print("\nSample POS id2label mapping:", dict(list(pos_id2label.items())[:5]))
print("Sample Chunk id2label mapping:", dict(list(chunk_id2label.items())[:5]))

Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


POS Model   → num_labels: 47
Chunk Model → num_labels: 23

Sample POS id2label mapping: {0: '"', 1: "''", 2: '#', 3: '$', 4: '('}
Sample Chunk id2label mapping: {0: 'O', 1: 'B-ADJP', 2: 'I-ADJP', 3: 'B-ADVP', 4: 'I-ADVP'}


---
## Task 4: Training (20%)

We use Hugging Face's `Trainer` API which handles:
- Gradient computation and backpropagation
- Learning rate scheduling
- Evaluation per epoch
- Saving best model checkpoints

**Hyperparameters used:**
| Parameter | Value |
|---|---|
| Learning Rate | 2e-5 |
| Epochs | 3 |
| Train Batch Size | 16 |
| Eval Batch Size | 16 |

In [15]:
import evaluate

def make_compute_metrics(label_list):
    """
    Returns a compute_metrics function that evaluates predictions
    using the seqeval metric (designed for sequence labeling tasks).
    """
    def compute_metrics(p):
        predictions, labels = p

        # Convert logits to predicted label indices
        predictions = np.argmax(predictions, axis=2)

        # Remove ignored tokens (-100) and convert ids back to label strings
        true_predictions = [
            [label_list[p] for p, l in zip(prediction, label) if l != -100]
            for prediction, label in zip(predictions, labels)
        ]

        true_labels = [
            [label_list[l] for p, l in zip(prediction, label) if l != -100]
            for prediction, label in zip(predictions, labels)
        ]

        # Correctly load the seqeval metric using the evaluate library
        seqeval_metric = evaluate.load("seqeval")
        results = seqeval_metric.compute(
            predictions=true_predictions,
            references=true_labels
        )

        return {
            "precision": results["overall_precision"],
            "recall": results["overall_recall"],
            "f1": results["overall_f1"],
            "accuracy": results["overall_accuracy"],
        }

    return compute_metrics

In [16]:
# ── Define Training Arguments ──
def get_training_args(output_dir):
    """
    Returns TrainingArguments with fixed hyperparameters.
    """
    return TrainingArguments(
        output_dir=output_dir,        # Directory to save checkpoints
        learning_rate=2e-5,           # Fine-tuning learning rate (small to avoid catastrophic forgetting)
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=3,           # 3 epochs is standard for fine-tuning
        weight_decay=0.01,            # L2 regularization
        evaluation_strategy="epoch",  # Evaluate at end of each epoch
        save_strategy="epoch",
        load_best_model_at_end=True,  # Keep the best model based on eval metric
        logging_steps=100,
        push_to_hub=False,
        report_to="none"              # Disable W&B logging
    )

print("TrainingArguments factory defined.")

TrainingArguments factory defined.


In [ ]:
# ─────────────────────────────────────────────
# TRAIN MODEL A: POS Tagging
# ─────────────────────────────────────────────

# Initialize the data collator
from transformers import DataCollatorForTokenClassification
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

pos_trainer = Trainer(
    model=pos_model,
    args=get_training_args("./pos-model"),
    train_dataset=pos_tokenized_dataset["train"],
    eval_dataset=pos_tokenized_dataset["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=make_compute_metrics(pos_label_list),
)

print("Starting POS Tagging training...")
pos_trainer.train()
print("POS Tagging training complete!")

Starting POS Tagging training...


Epoch,Training Loss,Validation Loss


In [ ]:
# ─────────────────────────────────────────────
# TRAIN MODEL B: Chunking
# ─────────────────────────────────────────────

chunk_trainer = Trainer(
    model=chunk_model,
    args=get_training_args("./chunk-model"),
    train_dataset=chunk_tokenized_dataset["train"],
    eval_dataset=chunk_tokenized_dataset["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=make_compute_metrics(chunk_label_list),
)

print("Starting Chunking training...")
chunk_trainer.train()
print("Chunking training complete!")

---
## Task 5: Evaluation (15%)

We evaluate both models on the **test split** using the `seqeval` metric.

**seqeval** is designed specifically for sequence labeling tasks like POS tagging and chunking.
It treats each entity/phrase as a complete unit and reports:
- **Precision:** Of all predicted labels, how many are correct?
- **Recall:** Of all actual labels, how many were correctly predicted?
- **F1 Score:** Harmonic mean of Precision and Recall.

In [ ]:
# ─────────────────────────────────────────────
# TASK 5: Evaluation on Test Set
# ─────────────────────────────────────────────

# Evaluate POS Tagging model
print("=" * 50)
print("POS TAGGING – Test Set Evaluation")
print("=" * 50)
pos_results = pos_trainer.evaluate(eval_dataset=pos_tokenized_dataset["test"])
print(f"Precision : {pos_results['eval_precision']:.4f}")
print(f"Recall    : {pos_results['eval_recall']:.4f}")
print(f"F1 Score  : {pos_results['eval_f1']:.4f}")
print(f"Accuracy  : {pos_results['eval_accuracy']:.4f}")

print()

# Evaluate Chunking model
print("=" * 50)
print("CHUNKING – Test Set Evaluation")
print("=" * 50)
chunk_results = chunk_trainer.evaluate(eval_dataset=chunk_tokenized_dataset["test"])
print(f"Precision : {chunk_results['eval_precision']:.4f}")
print(f"Recall    : {chunk_results['eval_recall']:.4f}")
print(f"F1 Score  : {chunk_results['eval_f1']:.4f}")
print(f"Accuracy  : {chunk_results['eval_accuracy']:.4f}")

---
## Task 6: Inference (10%)

We test both models on a custom sentence to see predictions.

In [ ]:
# ─────────────────────────────────────────────
# TASK 6: Inference on Custom Sentences
# ─────────────────────────────────────────────

def predict_tags(sentence, model, tokenizer, id2label):
    """
    Predicts token-level labels for a given sentence.

    Args:
        sentence  : Input text string.
        model     : Fine-tuned token classification model.
        tokenizer : Corresponding tokenizer.
        id2label  : Mapping from label id to label string.

    Returns:
        List of (word, predicted_label) tuples.
    """
    # Split sentence into words
    words = sentence.split()

    # Tokenize with offset mapping to track word boundaries
    inputs = tokenizer(
        words,
        is_split_into_words=True,
        return_tensors="pt",
        truncation=True
    )

    word_ids = inputs.word_ids()  # Maps subword tokens back to word indices

    # Run inference (no gradient computation needed)
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)

    # Get predicted label ids
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=2)[0].tolist()

    # Collect one prediction per original word (ignore subwords and special tokens)
    result = []
    prev_word_id = None
    for token_idx, word_id in enumerate(word_ids):
        if word_id is None:
            continue  # Skip [CLS] and [SEP]
        if word_id != prev_word_id:
            # First subword of a new word → record the prediction
            result.append((words[word_id], id2label[predictions[token_idx]]))
        prev_word_id = word_id

    return result


# ── Test Sentence ──
test_sentence = "John works at Google in California"
print(f"Input Sentence: {test_sentence}")
print("=" * 55)

# POS Tagging predictions
pos_predictions = predict_tags(test_sentence, pos_model, tokenizer, pos_id2label)
print("\nPOS Tag Predictions:")
print(f"{'Word':<20} {'POS Tag'}")
print("-" * 35)
for word, tag in pos_predictions:
    print(f"{word:<20} {tag}")

# Chunking predictions
chunk_predictions = predict_tags(test_sentence, chunk_model, tokenizer, chunk_id2label)
print("\nChunk Tag Predictions:")
print(f"{'Word':<20} {'Chunk Tag'}")
print("-" * 35)
for word, tag in chunk_predictions:
    print(f"{word:<20} {tag}")

In [ ]:
# ── Additional test sentences ──
more_sentences = [
    "The quick brown fox jumps over the lazy dog",
    "Apple released a new iPhone last September",
    "She is studying machine learning at MIT"
]

for sent in more_sentences:
    print(f"\nSentence: '{sent}'")
    print("-" * 60)
    pos_preds   = predict_tags(sent, pos_model,   tokenizer, pos_id2label)
    chunk_preds = predict_tags(sent, chunk_model, tokenizer, chunk_id2label)
    print(f"{'Word':<25} {'POS Tag':<15} {'Chunk Tag'}")
    print("-" * 55)
    for (word, ptag), (_, ctag) in zip(pos_preds, chunk_preds):
        print(f"{word:<25} {ptag:<15} {ctag}")

---
## Task 7: Comparison – POS Tagging vs Chunking (10%)

### Side-by-Side Comparison

| Aspect | POS Tagging | Chunking |
|---|---|---|
| **Level** | Word-level (Grammar) | Phrase-level (Structure) |
| **Output** | One tag per word | BIO-tagged phrase groups |
| **Task difficulty** | Easier | Harder |
| **Labels** | Fine-grained (40+ tags) | Coarser (Noun/Verb phrases) |
| **Use case** | Grammar checking, parsing | Information extraction, parsing |
| **Example label** | NN (noun), VBZ (verb) | B-NP (beginning of noun phrase) |

### Analysis from Results
- **POS Tagging** typically achieves higher F1 because labels are discrete and consistent.
- **Chunking** is harder due to phrase boundaries and context dependency.
- Both tasks benefit from BERT's bidirectional context.

In [ ]:
# ─────────────────────────────────────────────
# TASK 7: Quantitative Comparison
# ─────────────────────────────────────────────

print("="*55)
print("     COMPARISON: POS Tagging vs Chunking")
print("="*55)
print(f"{'Metric':<15} {'POS Tagging':>15} {'Chunking':>15}")
print("-"*45)
print(f"{'Precision':<15} {pos_results['eval_precision']:>15.4f} {chunk_results['eval_precision']:>15.4f}")
print(f"{'Recall':<15} {pos_results['eval_recall']:>15.4f} {chunk_results['eval_recall']:>15.4f}")
print(f"{'F1 Score':<15} {pos_results['eval_f1']:>15.4f} {chunk_results['eval_f1']:>15.4f}")
print(f"{'Accuracy':<15} {pos_results['eval_accuracy']:>15.4f} {chunk_results['eval_accuracy']:>15.4f}")
print("="*55)
print()
print("Observations:")
if pos_results['eval_f1'] > chunk_results['eval_f1']:
    diff = pos_results['eval_f1'] - chunk_results['eval_f1']
    print(f"  - POS Tagging outperforms Chunking by {diff:.4f} in F1 Score")
else:
    diff = chunk_results['eval_f1'] - pos_results['eval_f1']
    print(f"  - Chunking outperforms POS Tagging by {diff:.4f} in F1 Score")

print("  - POS Tagging: grammar-level labels per word (finer-grained)")
print("  - Chunking: phrase-level grouping requires context across words (harder)")

---
## Task 8: Report / Blog (5%)

### Understanding POS Tagging and Chunking with BERT

---

#### 1. What is POS Tagging?

Part-of-Speech (POS) tagging assigns a grammatical category to each word in a sentence — such as noun (NN), verb (VBZ), adjective (JJ), preposition (IN), etc. It works at the **word level**, labeling each token individually based on its role in the sentence.

For example:
```
John    → NNP  (Proper Noun)
works   → VBZ  (Verb, 3rd person singular)
at      → IN   (Preposition)
Google  → NNP  (Proper Noun)
```

---

#### 2. What is Chunking?

Chunking (also called shallow parsing) groups related words into **phrases** — Noun Phrases (NP), Verb Phrases (VP), Prepositional Phrases (PP), etc. It uses the BIO tagging scheme:
- **B-XX**: Beginning of a phrase of type XX
- **I-XX**: Inside a phrase (continuation)
- **O**: Outside any phrase

For example:
```
John    → B-NP  (Start of Noun Phrase)
works   → B-VP  (Start of Verb Phrase)
at      → B-PP  (Start of Prepositional Phrase)
Google  → B-NP  (Start of Noun Phrase)
in      → B-PP
California → B-NP
```

---

#### 3. Key Differences

| | POS Tagging | Chunking |
|--|--|--|
| **Granularity** | Word-level | Phrase-level |
| **Output scheme** | Single flat label per word | BIO scheme (spans across words) |
| **Task complexity** | Lower | Higher |
| **Information captured** | Grammatical role | Syntactic structure |
| **Common use** | Grammar checking, MT | IE, QA, parsing pipelines |

---

#### 4. Challenges Faced

**Subword Tokenization:**  
BERT's WordPiece tokenizer breaks rare or compound words into subwords (e.g., "California" → ["California"] but "Californian" → ["Californ", "##ian"]). This creates a mismatch between the number of tokens and labels. We resolved this by assigning `-100` to subword tokens so they are ignored in loss computation.

**BIO Label Coherence for Chunking:**  
The BIO scheme requires the model to understand multi-token spans. If the model predicts `I-NP` without a preceding `B-NP`, it's an invalid tag sequence. BERT handles this well due to bidirectional context, but it remains a harder task than flat POS tagging.

**Special Tokens:**  
`[CLS]` and `[SEP]` tokens added by BERT have no corresponding labels. We mask them with `-100` to exclude them from loss and metric computation.

---

#### 5. Observations & Insights

- **BERT's bidirectional context** is particularly helpful for chunking, where phrase boundaries depend on surrounding words.
- **DistilBERT** (used here) is 40% smaller and 60% faster than BERT while retaining ~97% of BERT's performance — making it ideal for this assignment.
- **POS tagging** converges faster and achieves higher accuracy due to simpler, context-independent labels.
- **Chunking** benefits from more epochs and may require a larger model or CRF layer on top of BERT for optimal span detection.
- The `seqeval` library correctly handles span-level evaluation, treating complete phrases as units rather than individual tokens.

---

#### 6. Conclusion

Fine-tuning DistilBERT for token classification is a powerful and relatively straightforward approach to POS tagging and chunking. The Hugging Face ecosystem (datasets, transformers, evaluate, Trainer API) makes the entire pipeline — from data loading to evaluation — reproducible and clean. The key challenge lies in correct label alignment with subword tokens, which is critical for model quality.

---
## Summary: Full Pipeline

```
Raw Data (CoNLL-2003)
       ↓
Tokenization (DistilBERT WordPiece)
       ↓
Label Alignment (subwords → -100, first subword → true label)
       ↓
Model Training (Hugging Face Trainer, 3 epochs)
       ↓
Evaluation (seqeval: Precision, Recall, F1)
       ↓
Inference (custom sentences)
       ↓
Comparison (POS Tagging vs Chunking)
```

**Models trained:**
- `./pos-model/` → POS Tagging model
- `./chunk-model/` → Chunking model